# LangChain Using Tools

In [1]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain.tools import tool
from langchain_core.output_parsers.openai_tools import parse_tool_calls
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = "gpt-5.6-sol"

In [4]:
llm = ChatOpenAI(
    model=model,
    temperature=0.0,
    max_tokens=500,
    timeout=500,
    reasoning_effort="low",
    max_retries=2,
)

In [5]:
@tool
def add(a: int, b: int) -> int:
    """Add two numbers.

    Args:
        a (int): The first number.
        b (int): The second number.
    """
    return a + b

In [6]:
@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers.

    Args:
        a (int): The first number.
        b (int): The second number.
    """
    return a * b

In [7]:
tools = [add, multiply]
tool_map = {tool.name: tool for tool in tools}

In [8]:
tool_map

{'add': StructuredTool(name='add', description='Add two numbers.\n\nArgs:\n    a (int): The first number.\n    b (int): The second number.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x00000229F6CD75E0>),
 'multiply': StructuredTool(name='multiply', description='Multiply two numbers.\n\nArgs:\n    a (int): The first number.\n    b (int): The second number.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x00000229F6CD7690>)}

In [9]:
print(tool_map)

{'add': StructuredTool(name='add', description='Add two numbers.\n\nArgs:\n    a (int): The first number.\n    b (int): The second number.', args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x00000229F6CD75E0>), 'multiply': StructuredTool(name='multiply', description='Multiply two numbers.\n\nArgs:\n    a (int): The first number.\n    b (int): The second number.', args_schema=<class 'langchain_core.utils.pydantic.multiply'>, func=<function multiply at 0x00000229F6CD7690>)}


In [10]:
llm_with_tools = llm.bind_tools(tools, strict=True)

In [11]:
question = "What is the sum of 5 and 10, and then multiply the result by 2?"

In [12]:
messages = [
    SystemMessage(
        """
            You're a helpful assistant that can perform mathematical operations using the provided tools only.
            You can only use the tools provided and cannot perform any calculations on your own.
            If you need to perform a calculation, you must use the appropriate tool.
            You can use the following tools:
            1. add: Adds two numbers together.
            2. multiply: Multiplies two numbers together.
            When you need to use a tool, you must call it in the following format:
            tool_name(arg1, arg2)
        """
    ),
    HumanMessage(question),
]

In [13]:
ai_message = llm_with_tools.invoke(messages)

In [14]:
ai_message

AIMessage(content=[{'id': 'rs_0dd182ee7246105f006ab3aedb438487d28f69ff4772b31585', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqs67bdquOlzYYuwxDCxwZdxoib1jm3GBuHXQNaiuBeqhcxQ0UlMTZsx5YuPiIscOP_e361ZFnHVEUnxphvC9buD_79XJuNsbXGZbS2JOd5y4TexPWTdX1idxSFQrfvNJajd6ACnOlmICB4mLKSTSR1P9FXLRSMcu12uTLH1VrkV3eHY60YSKR3UE4KYUHDzOIHWLWTIk0VfUlwcr4zTr5vBaNJrsRU9jy5YCz9NxJ0cj7K-jqmfISKHOIiLa9uul-h3Uc5zIvPryH7Q5FtcsNx0c-FQXSjO-Ibhp_kui-XFIHGZNfXzioJzFB4FtZx8ny8sXXMLACUS6aJ4sgSVOFadzndPOK49C4L5kSuBbuYDiOSmIP8h5uFyp1m1EP7J5cjfCryH57JpM4tyu0wqb1uEgXSukst4w4LCdT-CVlKQFyZ33O_uW-4PJptZUr5yFydruDHVvMWixZSVpXhFw3RVX1v-q5ENdgAPewAeCfILWCre-4lHbFeK18hEfWYGlk4tdry8W29q05E5XrbmcV7z-Av4bvw4YdXf9b1eN9o0jVrWcjbUUfRNar9PEt6Pgg-jgwxJpN6LMxaczc6kEQyCtk3rDWe8cRCmFySx_tmKMFPYZ4gyDniydhK6SnSwn4Ifg6qysau01RM3O9jazNqPit7R0296VIJHTd2U2cojpjR2elkJwB0qTm-gvgIm8HRvklpw2gvr5wDo3xzQcUkMt7o6ohul5RWs8TXuwyDAcP54LJFZP-K4NtGmTyMbfHlBFngtLFWSPpusDFD5y2MKLFoBX2IxJY0Qx1-fmXvmZjZrXoI-H_77jkb3kASW48jmxj

In [15]:
ai_message.tool_calls

[{'name': 'add',
  'args': {'a': 5, 'b': 10},
  'id': 'call_dAUqNK0K7R1AEOB7DY6dYH3Q',
  'type': 'tool_call'}]

In [16]:
messages.append(ai_message)

In [17]:
messages

[SystemMessage(content="\n            You're a helpful assistant that can perform mathematical operations using the provided tools only.\n            You can only use the tools provided and cannot perform any calculations on your own.\n            If you need to perform a calculation, you must use the appropriate tool.\n            You can use the following tools:\n            1. add: Adds two numbers together.\n            2. multiply: Multiplies two numbers together.\n            When you need to use a tool, you must call it in the following format:\n            tool_name(arg1, arg2)\n        ", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the sum of 5 and 10, and then multiply the result by 2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'id': 'rs_0dd182ee7246105f006ab3aedb438487d28f69ff4772b31585', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqs67bdquOlzYYuwxDCxwZdxoib1jm3GBuHXQNaiuBeqhcxQ0UlMT

In [18]:
for message in messages:
    message.pretty_print()

================================ System Message ================================


            You're a helpful assistant that can perform mathematical operations using the provided tools only.
            You can only use the tools provided and cannot perform any calculations on your own.
            If you need to perform a calculation, you must use the appropriate tool.
            You can use the following tools:
            1. add: Adds two numbers together.
            2. multiply: Multiplies two numbers together.
            When you need to use a tool, you must call it in the following format:
            tool_name(arg1, arg2)
        
================================ Human Message =================================

What is the sum of 5 and 10, and then multiply the result by 2?
================================== Ai Message ==================================

[{'id': 'rs_0dd182ee7246105f006ab3aedb438487d28f69ff4772b31585', 'summary': [], 'type': 'reasoning', 'content': [], 'enc

In [20]:
# parsed_tool_calls = parse_tool_calls(ai_message.additional_kwargs.get("tool_calls"))

In [21]:
# parsed_tool_calls

In [22]:
for tool_call in ai_message.tool_calls:
    tool_call_id = tool_call["id"]
    function_name = tool_call["name"]
    arguments = tool_call["args"]
    func = tool_map[function_name]
    result = func.invoke(arguments)
    tool_message = ToolMessage(
        content=result,
        name=function_name,
        tool_call_id=tool_call_id,
    )
    messages.append(tool_message)

In [23]:
messages

[SystemMessage(content="\n            You're a helpful assistant that can perform mathematical operations using the provided tools only.\n            You can only use the tools provided and cannot perform any calculations on your own.\n            If you need to perform a calculation, you must use the appropriate tool.\n            You can use the following tools:\n            1. add: Adds two numbers together.\n            2. multiply: Multiplies two numbers together.\n            When you need to use a tool, you must call it in the following format:\n            tool_name(arg1, arg2)\n        ", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the sum of 5 and 10, and then multiply the result by 2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'id': 'rs_0dd182ee7246105f006ab3aedb438487d28f69ff4772b31585', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqs67bdquOlzYYuwxDCxwZdxoib1jm3GBuHXQNaiuBeqhcxQ0UlMT

In [24]:
for message in messages:
    message.pretty_print()

================================ System Message ================================


            You're a helpful assistant that can perform mathematical operations using the provided tools only.
            You can only use the tools provided and cannot perform any calculations on your own.
            If you need to perform a calculation, you must use the appropriate tool.
            You can use the following tools:
            1. add: Adds two numbers together.
            2. multiply: Multiplies two numbers together.
            When you need to use a tool, you must call it in the following format:
            tool_name(arg1, arg2)
        
================================ Human Message =================================

What is the sum of 5 and 10, and then multiply the result by 2?
================================== Ai Message ==================================

[{'id': 'rs_0dd182ee7246105f006ab3aedb438487d28f69ff4772b31585', 'summary': [], 'type': 'reasoning', 'content': [], 'enc

In [25]:
ai_message = llm_with_tools.invoke(messages)

In [26]:
ai_message

AIMessage(content=[{'arguments': '{"a":15,"b":2}', 'call_id': 'call_jnycF9OMnwO6ivR8fGtWvWNl', 'name': 'multiply', 'type': 'function_call', 'id': 'fc_0dd182ee7246105f006ab3af6106fc87d2ae6b2292896d3268', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0dd182ee7246105f006ab3af5edbec87d293428c6e67469cb3', 'created_at': 1790160735.0, 'metadata': {}, 'model': 'gpt-5.6-sol', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.6-sol'}, id='resp_0dd182ee7246105f006ab3af5edbec87d293428c6e67469cb3', tool_calls=[{'name': 'multiply', 'args': {'a': 15, 'b': 2}, 'id': 'call_jnycF9OMnwO6ivR8fGtWvWNl', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 277, 'output_tokens': 21, 'total_tokens': 298, 'input_token_details': {'cache_creation': 0, 'cache_read': 0}, 'output_token_details': {'reasoning': 0}})

In [27]:
for tool_call in ai_message.tool_calls:
    tool_call_id = tool_call["id"]
    function_name = tool_call["name"]
    arguments = tool_call["args"]
    func = tool_map[function_name]
    result = func.invoke(arguments)
    tool_message = ToolMessage(
        content=result,
        name=function_name,
        tool_call_id=tool_call_id,
    )
    messages.append(tool_message)

In [28]:
for message in messages:
    message.pretty_print()

================================ System Message ================================


            You're a helpful assistant that can perform mathematical operations using the provided tools only.
            You can only use the tools provided and cannot perform any calculations on your own.
            If you need to perform a calculation, you must use the appropriate tool.
            You can use the following tools:
            1. add: Adds two numbers together.
            2. multiply: Multiplies two numbers together.
            When you need to use a tool, you must call it in the following format:
            tool_name(arg1, arg2)
        
================================ Human Message =================================

What is the sum of 5 and 10, and then multiply the result by 2?
================================== Ai Message ==================================

[{'id': 'rs_0dd182ee7246105f006ab3aedb438487d28f69ff4772b31585', 'summary': [], 'type': 'reasoning', 'content': [], 'enc

In [30]:
messages

[SystemMessage(content="\n            You're a helpful assistant that can perform mathematical operations using the provided tools only.\n            You can only use the tools provided and cannot perform any calculations on your own.\n            If you need to perform a calculation, you must use the appropriate tool.\n            You can use the following tools:\n            1. add: Adds two numbers together.\n            2. multiply: Multiplies two numbers together.\n            When you need to use a tool, you must call it in the following format:\n            tool_name(arg1, arg2)\n        ", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What is the sum of 5 and 10, and then multiply the result by 2?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[{'id': 'rs_0dd182ee7246105f006ab3aedb438487d28f69ff4772b31585', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqs67bdquOlzYYuwxDCxwZdxoib1jm3GBuHXQNaiuBeqhcxQ0UlMT